In [ ]:
!pip install -q "numpy<2" "faiss-cpu==1.8.0" --force-reinstall --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 84.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
print("HF token loaded")

HF token loaded


In [ ]:
# Verify NumPy is now 1.x
import numpy
print("NumPy version:", numpy.__version__)

NumPy version: 1.26.4


In [ ]:
!cp /content/drive/MyDrive/retriever.py /content/retriever.py

import sys
sys.path.insert(0, "/content")

from retriever import load_retriever, retrieve_passages

print("Loading Shruti's retriever...")
load_retriever()
print("Retriever ready")

results = retrieve_passages("Who wrote Hamlet?", k=3)
for i, p in enumerate(results, 1):
    print(f"[{i}] (score={p['score']:.3f}) {p['text'][:150]}")

Loading Shruti's retriever...
Loading sentence-transformers embedder...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Loading Wikipedia passage corpus...


Generating passages split:   0%|          | 0/3200 [00:00<?, ? examples/s]

Encoding 3200 passages...


Batches:   0%|          | 0/50 [00:00<?, ?it/s]

Retriever ready. Index has 3200 passages of dim 384.
Retriever ready
[1] (score=0.403) Isaac Newton in 1712. Portrait by Sir James Thornhill.
[2] (score=0.390) Portrait of Blaise Pascal
[3] (score=0.390) John Adams, portrait by John Trumbull.


In [ ]:
!pip install -q -U bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading Mistral-7B-Instruct tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading Mistral-7B-Instruct model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()
print("Mistral loaded")

Loading Mistral-7B-Instruct tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Mistral loaded


In [ ]:
def generate_answer(prompt, max_new_tokens=60):
    """Greedy generation from Mistral."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3072).to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
def build_rag_prompt(question, passages):
    """
    Construct a RAG prompt that gives Mistral the question + retrieved passages
    and asks it to answer using the passages.
    """
    passages_text = "\n\n".join(
        [f"[Passage {i+1}] {p['text']}" for i, p in enumerate(passages)]
    )
    return (
        f"[INST] You are given several passages from Wikipedia that may contain "
        f"the answer to a question. Use the passages to answer the question with "
        f"a short, concise answer. If the passages do not contain the answer, "
        f"answer based on general knowledge.\n\n"
        f"Passages:\n{passages_text}\n\n"
        f"Question: {question} [/INST] Answer:"
    )

# Sanity-check the prompt
sample_q = "Who wrote the play Hamlet?"
sample_passages = retrieve_passages(sample_q, k=3)
print(build_rag_prompt(sample_q, sample_passages)[:1000])
print("...")

[INST] You are given several passages from Wikipedia that may contain the answer to a question. Use the passages to answer the question with a short, concise answer. If the passages do not contain the answer, answer based on general knowledge.

Passages:
[Passage 1] Portrait of Blaise Pascal

[Passage 2] Isaac Newton in 1712. Portrait by Sir James Thornhill.

[Passage 3] A discussion of Pascal figures prominently in the movie My Night At Maud's by the French director Ãric Rohmer.

Question: Who wrote the play Hamlet? [/INST] Answer:
...


In [ ]:
def rag_answer(question, k=5, return_passages=False):
    """
    End-to-end RAG: retrieve k passages, then generate an answer using them.

    Args:
        question: Input question.
        k: Number of passages to retrieve.
        return_passages: If True, also return the retrieved passages.

    Returns:
        Answer string, or (answer, passages) tuple if return_passages=True.
    """
    passages = retrieve_passages(question, k=k)
    prompt = build_rag_prompt(question, passages)
    answer = generate_answer(prompt, max_new_tokens=60)
    if return_passages:
        return answer, passages
    return answer

# Test it!
print("="*60)
print("RAG PIPELINE END-TO-END TEST")
print("="*60)
for q in ["Who wrote the play Hamlet?", "What is the capital of Australia?", "In which year did the Titanic sink?"]:
    answer, passages = rag_answer(q, k=5, return_passages=True)
    print(f"\nQ: {q}")
    print(f"Retrieved titles: {[p['title'] for p in passages[:3]]}")
    print(f"RAG answer: {answer[:200]}")

RAG PIPELINE END-TO-END TEST

Q: Who wrote the play Hamlet?
Retrieved titles: ['Passage 223', 'Passage 562', 'Passage 268']
RAG answer: William Shakespeare wrote the play Hamlet. (This question is not related to the given passages.)

Q: What is the capital of Australia?
Retrieved titles: ['Passage 1723', 'Passage 36', 'Passage 809']
RAG answer: Australia does not have a capital city as such, but the kangaroo is a national icon and is featured on its coat of arms, currency, and is used by many Australian organizations.

Q: In which year did the Titanic sink?
Retrieved titles: ['Passage 2159', 'Passage 3012', 'Passage 2962']
RAG answer: The RMS Titanic sank in 1912. (This question is not directly answered in the passages, but it is well-known information.)


In [ ]:
from datasets import load_dataset
import random
import json
from tqdm import tqdm

print("Loading TriviaQA validation set...")
triviaqa_val = load_dataset("trivia_qa", "rc.nocontext", split="validation")

# Use same seed as Mayur but take first 30 of the 500-slice for consistency
random.seed(42)
eval_indices = random.sample(range(len(triviaqa_val)), 500)
demo_set = triviaqa_val.select(eval_indices[:30])

print(f"Running RAG on {len(demo_set)} demo questions...")

rag_results = []
for ex in tqdm(demo_set, desc="RAG"):
    question = ex["question"]
    gold = ex["answer"]["value"]
    aliases = ex["answer"]["aliases"]

    answer, passages = rag_answer(question, k=5, return_passages=True)

    rag_results.append({
        "question": question,
        "gold": gold,
        "aliases": aliases,
        "prediction": answer,
        "retrieved_titles": [p["title"] for p in passages],
        "top_passage_score": passages[0]["score"] if passages else 0.0,
    })

with open("/content/drive/MyDrive/rag_demo_results.json", "w") as f:
    json.dump(rag_results, f, indent=2)

print(f"\n Saved {len(rag_results)} RAG predictions to Drive.")

# Show first 5 with verdict
import re, string
def normalize(s):
    s = s.lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = "".join(c for c in s if c not in string.punctuation)
    return " ".join(s.split())

def is_correct(pred, gold, aliases):
    pred_n = normalize(pred)
    return any(normalize(c) in pred_n for c in [gold] + list(aliases) if normalize(c))

correct = sum(is_correct(r["prediction"], r["gold"], r["aliases"]) for r in rag_results)
print(f"\nLoose EM on 30 demo questions: {correct}/30 = {correct/30:.1%}")

print("\nFirst 5 predictions:")
for r in rag_results[:5]:
    mark = "✅" if is_correct(r["prediction"], r["gold"], r["aliases"]) else "❌"
    print(f"\n{mark} Q: {r['question'][:90]}")
    print(f"   Gold: {r['gold']}")
    print(f"   Pred: {r['prediction'][:120]}")
    print(f"   Top score: {r['top_passage_score']:.3f}")

Loading TriviaQA validation set...


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/138384 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/17944 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/17210 [00:00<?, ? examples/s]

Running RAG on 30 demo questions...


RAG: 100%|██████████| 30/30 [02:10<00:00,  4.36s/it]


 Saved 30 RAG predictions to Drive.

Loose EM on 30 demo questions: 18/30 = 60.0%

First 5 predictions:

❌ Q: What was first worn by British soldiers in India in 1845?
   Gold: Khaki
   Pred: Ford, who is not a British soldier but the founder of the Ford Motor Company in 1891, is depicted in Passage 1 wearing a
   Top score: 0.417

❌ Q: Which US gangster was released from Alcatraz prison in November 1939?
   Gold: Al Capone
   Pred: John Wilkes Booth, who is known for assassinating President Abraham Lincoln in 1865, is not the US gangster mentioned in
   Top score: 0.392

✅ Q: Of which tribe of Red Indians was Geronimo a chief
   Gold: Apache
   Pred: Geronimo was a chief of the Southwestern Apache tribe.
   Top score: 0.569

❌ Q: Which rank of the RAF is equivalent to Major in the British Army?
   Gold: Squadron Leader
   Pred: The rank of Wing Commander in the Royal Air Force is equivalent to Major in the British Army. However, the passages prov
   Top score: 0.374

✅ Q: Name the ca

In [ ]:
rag_code = '''
"""
rag_pipeline.py — End-to-end Retrieval-Augmented Generation pipeline
Author: Satyam Kale
DATA 612 Group Project

Stitches together:
  - Shruti's retriever (sentence-transformers + FAISS over mini-wikipedia)
  - Mistral-7B-Instruct-v0.2 generator (4-bit quantized)
"""

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from retriever import load_retriever, retrieve_passages

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

_model = None
_tokenizer = None


def load_generator():
    """Load Mistral-7B-Instruct in 4-bit."""
    global _model, _tokenizer
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    _tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if _tokenizer.pad_token is None:
        _tokenizer.pad_token = _tokenizer.eos_token
    _model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    _model.eval()


def generate_answer(prompt, max_new_tokens=60):
    """Greedy generation from Mistral."""
    if _model is None:
        load_generator()
    inputs = _tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=3072
    ).to(_model.device)
    with torch.no_grad():
        output = _model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=_tokenizer.pad_token_id,
        )
    return _tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()


def build_rag_prompt(question, passages):
    """Build a RAG prompt with question + retrieved passages."""
    passages_text = "\\n\\n".join(
        [f"[Passage {i+1}] {p[\\"text\\"]}" for i, p in enumerate(passages)]
    )
    return (
        f"[INST] You are given several passages from Wikipedia that may contain "
        f"the answer to a question. Use the passages to answer the question with "
        f"a short, concise answer. If the passages do not contain the answer, "
        f"answer based on general knowledge.\\n\\n"
        f"Passages:\\n{passages_text}\\n\\n"
        f"Question: {question} [/INST] Answer:"
    )


def rag_answer(question, k=5, return_passages=False):
    """End-to-end RAG: retrieve k passages then generate an answer."""
    if _model is None:
        load_generator()
    passages = retrieve_passages(question, k=k)
    prompt = build_rag_prompt(question, passages)
    answer = generate_answer(prompt)
    if return_passages:
        return answer, passages
    return answer


if __name__ == "__main__":
    load_retriever()
    load_generator()
    for q in ["Who wrote the play Hamlet?", "What is the capital of Australia?"]:
        a, p = rag_answer(q, k=5, return_passages=True)
        print(f"Q: {q}")
        print(f"Retrieved: {[x[\\"title\\"] for x in p[:3]]}")
        print(f"A: {a}\\n")
'''

with open('/content/drive/MyDrive/rag_pipeline.py', 'w') as f:
    f.write(rag_code)

print("Saved rag_pipeline.py to Google Drive")

Saved rag_pipeline.py to Google Drive
